# Stability checks: leaf size, seeds, calibration

Run from the repository root, with the raw OULAD files in `raw/` (see README).

In [ ]:
import os
from pathlib import Path

root = Path.cwd()
for p in [root, *root.parents]:
    if (p / "raw").is_dir():
        root = p
        break
BASE = str(root)

In [2]:
!pip install econml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3/155.3 kB 12.1 MB/s eta 0:00:00


In [5]:
import sys, os, json, time
import numpy as np
import pandas as pd
from econml.dml import CausalForestDML, LinearDML
from sklearn.ensemble import HistGradientBoostingRegressor

import econml, sklearn
print("econml:", econml.__version__, "| sklearn:", sklearn.__version__)
assert econml.__version__.startswith("0.17"), "VERIFY parameter names for your version"

CORPUS_PATH = f'{BASE}/oulad_full_corpus.csv'
SEED, N_TREES, CANONICAL_ATE = 42, 500, 0.004786
LEAF_GRID = [5, 15, 25, 50, 100]
SEEDS     = [42, 1, 7, 123, 2024]
N_PERM    = 100

CONFOUNDERS_V2 = ['region', 'imd_band', 'highest_education', 'age_band', 'gender',
                  'disability', 'code_presentation', 'num_of_prev_attempts',
                  'studied_credits', 'code_module']
CATEGORICAL_V2 = ['region', 'imd_band', 'highest_education', 'age_band', 'gender',
                  'disability', 'code_presentation', 'code_module']

full = pd.read_csv(CORPUS_PATH)
full['imd_band'] = full['imd_band'].fillna('Missing')
X = pd.get_dummies(full[CONFOUNDERS_V2], columns=CATEGORICAL_V2, drop_first=True).astype(float)
T = full['oucontent_clicks'].values.astype(float)
Y = full['performance_gain'].values.astype(float)
print(f"n = {len(full):,} | X = {X.shape}")
assert X.shape == (17529, 40), f"shape drift: {X.shape}"

def fit_cf(leaf, seed=SEED, y=Y, t=T, x=X):
    est = CausalForestDML(
        model_y=HistGradientBoostingRegressor(random_state=seed),
        model_t=HistGradientBoostingRegressor(random_state=seed),
        discrete_treatment=False, honest=True,
        min_samples_leaf=leaf, n_estimators=N_TREES, cv=5, random_state=seed)
    est.fit(y, t, X=x, cache_values=True)
    return est

print("PART A — min_samples_leaf SENSITIVITY STRIP")

t0 = time.time()
base = fit_cf(LEAF_GRID[0])
rows = []
for leaf in LEAF_GRID:
    if leaf != LEAF_GRID[0]:

        base.min_samples_leaf = leaf
        base.refit_final()
    ate = base.ate(X)
    lo, hi = base.ate_interval(X, alpha=0.05)
    cate = base.effect(X)
    rows.append({'min_samples_leaf': leaf, 'ATE': ate, 'CI_low': lo, 'CI_high': hi,
                 'CI_width': hi - lo, 'CATE_sd': cate.std(),
                 'pct_negative': (cate < 0).mean(),
                 'excludes_zero': 'yes' if lo > 0 or hi < 0 else 'no'})
    print(f"  leaf={leaf:>4}  ATE={ate:.6f}  CI=[{lo:.6f},{hi:.6f}]  "
          f"width={hi-lo:.6f}  neg={rows[-1]['pct_negative']:.1%}")

strip = pd.DataFrame(rows)

assert strip['ATE'].nunique() > 1, \
    "all rows identical -- min_samples_leaf did not propagate through refit_final()"
_r50 = strip.loc[strip['min_samples_leaf'] == 50, 'ATE']
if len(_r50):
    print(f"\n  leaf=50 row : {float(_r50.iloc[0])!r}")
    print(f"  canonical   : {CANONICAL_ATE}")
    print(f"  difference  : {abs(float(_r50.iloc[0]) - CANONICAL_ATE):.2e}")

strip.to_csv(f'{BASE}/leaf_sensitivity_strip.csv', index=False)
print(f"\n[{time.time()-t0:.0f}s]")
print(f"ATE range across the grid: {strip['ATE'].min():.6f} to {strip['ATE'].max():.6f}  "
      f"(spread {strip['ATE'].max()-strip['ATE'].min():.6f})")

print("PART B — MULTI-SEED REPLICATION")

t0, srows = time.time(), []
for s in SEEDS:
    cf_s = fit_cf(50, seed=s)
    a = cf_s.ate(X); lo, hi = cf_s.ate_interval(X, alpha=0.05)
    ld = LinearDML(model_y=HistGradientBoostingRegressor(random_state=s),
                   model_t=HistGradientBoostingRegressor(random_state=s),
                   discrete_treatment=False, cv=5, random_state=s)
    ld.fit(Y, T, X=X)
    la = ld.ate(X); llo, lhi = ld.ate_interval(X, alpha=0.05)
    srows.append({'seed': s, 'CF_ATE': a, 'CF_width': hi - lo,
                  'LDML_ATE': la, 'LDML_width': lhi - llo})
    print(f"  seed={s:>5}  CF={a:.6f} (w={hi-lo:.6f})   LDML={la:.6f} (w={lhi-llo:.6f})")

sd = pd.DataFrame(srows)
sd.to_csv(f'{BASE}/multiseed_replication.csv', index=False)
print(f"\n[{time.time()-t0:.0f}s]")
print(f"CF   ATE across seeds: mean={sd['CF_ATE'].mean():.6f} +/- {sd['CF_ATE'].std():.6f}"
      f"  [{sd['CF_ATE'].min():.6f}, {sd['CF_ATE'].max():.6f}]")
print(f"LDML ATE across seeds: mean={sd['LDML_ATE'].mean():.6f} +/- {sd['LDML_ATE'].std():.6f}")
print(f"\nSeed SD as a share of the analytic CI width: "
      f"{sd['CF_ATE'].std()/sd['CF_width'].mean():.1%}")

cf50 = fit_cf(50)
c_lo, c_hi = cf50.ate_interval(X, alpha=0.05)
ld = LinearDML(model_y=HistGradientBoostingRegressor(random_state=SEED),
               model_t=HistGradientBoostingRegressor(random_state=SEED),
               discrete_treatment=False, cv=5, random_state=SEED)
ld.fit(Y, T, X=X)
l_lo, l_hi = ld.ate_interval(X, alpha=0.05)
print(f"\nCF   CI: [{c_lo:.6f}, {c_hi:.6f}]")
print(f"LDML CI: [{l_lo:.6f}, {l_hi:.6f}]")
if c_lo <= l_lo and c_hi >= l_hi:
    print("  *** CF's ANALYTIC interval contains Linear DML's entirely.")
    print()

print("PART C — IS THE CATE SPREAD REAL, OR ESTIMATION NOISE?")

used_drtester = False
print()

if not used_drtester:
    t0 = time.time()
    obs_var = cf50.effect(X).var()
    ate_hat = cf50.ate(X)
    print(f"observed CATE variance = {obs_var:.4e}")

    resid = Y - ate_hat * T
    rng = np.random.default_rng(SEED)

    CKPT_C = f'{BASE}/s3_partC_progress.json'
    null_vars = []
    if os.path.exists(CKPT_C) and os.path.getsize(CKPT_C) > 0:
        try:
            null_vars = json.load(open(CKPT_C))['null_vars']
            print(f"resuming -- {len(null_vars)} of {N_PERM} draws already complete")
        except Exception:
            null_vars = []

    for b in range(len(null_vars), N_PERM):
        Y_null = ate_hat * T + rng.permutation(resid)
        cf_n = fit_cf(50, seed=SEED + b, y=Y_null)
        null_vars.append(float(cf_n.effect(X).var()))
        if (b + 1) % 10 == 0:
            json.dump({'null_vars': null_vars}, open(CKPT_C, 'w'))
            print(f"  {b+1}/{N_PERM}  [{time.time()-t0:.0f}s]  checkpointed")

    nv = np.array(null_vars)
    p = (1 + (nv >= obs_var).sum()) / (1 + len(nv))
    print(f"\nnull variance: mean={nv.mean():.4e}  sd={nv.std():.4e}  max={nv.max():.4e}")
    print(f"observed      : {obs_var:.4e}")
    print(f"empirical p = {p:.4f}   (n={len(nv)} draws; floor is {1/(1+len(nv)):.4f})")
    pd.DataFrame({'null_variance': nv}).to_csv(f'{BASE}/calibration_null.csv', index=False)

print("PART D — IS A FOREST WARRANTED ON AN ALL-CATEGORICAL COVARIATE SPACE?")

cf_ate = cf50.ate(X); cf_lo, cf_hi = cf50.ate_interval(X, alpha=0.05)
ld_ate = ld.ate(X)
print(f"CausalForestDML : ATE={cf_ate:.6f}  CI=[{cf_lo:.6f},{cf_hi:.6f}]  w={cf_hi-cf_lo:.6f}")
print(f"LinearDML       : ATE={ld_ate:.6f}  CI=[{l_lo:.6f},{l_hi:.6f}]  w={l_hi-l_lo:.6f}")
print(f"width ratio (CF / LDML) = {(cf_hi-cf_lo)/(l_hi-l_lo):.2f}x")

cmp = pd.DataFrame({'module': full['code_module'],
                    'cf': cf50.effect(X), 'ldml': ld.effect(X)}) \
        .groupby('module').mean()
cmp['abs_diff'] = (cmp['cf'] - cmp['ldml']).abs()
print("\nPer-module CATE, both estimators:")
print(cmp.to_string(float_format=lambda v: f"{v:.6f}"))
print(f"\ncorrelation of per-student CATE: "
      f"{np.corrcoef(cf50.effect(X), ld.effect(X))[0,1]:.4f}")
cmp.to_csv(f'{BASE}/modality_per_module.csv')


print("multiseed_replication.csv, calibration_null.csv, modality_per_module.csv")

econml: 0.17.0 | sklearn: 1.6.1
n = 17,529 | X = (17529, 40)

PART A — min_samples_leaf SENSITIVITY STRIP

  leaf=   5  ATE=0.004731  CI=[-0.001127,0.010588]  width=0.011715  neg=7.8%
  leaf=  15  ATE=0.004732  CI=[0.001358,0.008106]  width=0.006748  neg=6.5%
  leaf=  25  ATE=0.004758  CI=[0.002014,0.007501]  width=0.005487  neg=6.0%
  leaf=  50  ATE=0.004786  CI=[0.002576,0.006997]  width=0.004421  neg=4.5%
  leaf= 100  ATE=0.004810  CI=[0.002978,0.006642]  width=0.003664  neg=2.7%

  leaf=50 row : 0.004786482048864249
  canonical   : 0.004786
  difference  : 4.82e-07

[127s]
ATE range across the grid: 0.004731 to 0.004810  (spread 0.000079)


PART B — MULTI-SEED REPLICATION

  seed=   42  CF=0.004786 (w=0.004421)   LDML=0.003706 (w=0.001982)
  seed=    1  CF=0.004857 (w=0.004475)   LDML=0.003697 (w=0.001970)
  seed=    7  CF=0.004793 (w=0.004582)   LDML=0.003751 (w=0.001964)
  seed=  123  CF=0.004699 (w=0.004363)   LDML=0.003617 (w=0.001970)
  seed= 2024  CF=0.004803 (w=0.004182)   L